<a href="https://colab.research.google.com/github/GabrielvanderSchmidt/LangChain-Experiments/blob/main/retrieval/2_stage_ir_with_eval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# @title Setup Env
!pip install -q transformers datasets accelerate sentence-transformers \
                langchain langchain-huggingface langchain-chroma langchain-community \
                --upgrade

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 112.5/112.5 kB 8.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 71.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 70.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 35.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 19.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.9/64.9 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 74.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 

# 2-stage IR without legacy packages

This is a vibe-coded reimplementation of the original 2-stage IR pipeline without usage of legacy (`langchain_classic`) features, which is (allegedly) meant to be a more modern and interoperable take on the usage of HF rerankers inside LangChain.

While mostly just a proof of concept, this script should be useful as a template or reference for further experiments that may run into similar problems.

In [ ]:
import langchain, langchain_core
langchain.__version__, langchain_core.__version__

('1.2.12', '1.2.19')

In [ ]:
# Made with AdaptaOne
#
# =========================
# Two-stage Information Retrieval
# LangChain 1.2.x + Chroma + Hugging Face
# Stage 1: dense vector retrieval
# Stage 2: cross-encoder reranking
# =========================

from __future__ import annotations

from dataclasses import dataclass
from typing import List, Sequence, Optional, Dict, Any
import os

import torch
from sentence_transformers import CrossEncoder

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma


# ============================================================
# Configuration
# ============================================================

@dataclass
class RetrievalConfig:
    # Embedding model for stage 1
    embedding_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"

    # Cross-encoder reranker model for stage 2
    # Good lightweight default for limited consumer GPUs
    reranker_model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"

    # Chroma persistence
    persist_directory: str = "./chroma_langchain_db"
    collection_name: str = "basic_ir_collection"

    # Retrieval sizes
    initial_k: int = 20   # retrieved from vector store
    final_k: int = 5      # kept after reranking

    # Device and performance
    device: Optional[str] = None
    normalize_embeddings: bool = True

    # Cross-encoder inference batch size
    # Keep conservative for consumer GPU with limited VRAM
    reranker_batch_size: int = 8


# ============================================================
# Utility functions
# ============================================================

def get_device(explicit_device: Optional[str] = None) -> str:
    if explicit_device:
        return explicit_device
    return "cuda" if torch.cuda.is_available() else "cpu"


def print_ranked_results(
    docs: Sequence[Document],
    score_key: str = "reranker_score",
    max_chars: int = 400,
) -> None:
    if not docs:
        print("No documents found.")
        return

    for idx, doc in enumerate(docs, start=1):
        score = doc.metadata.get(score_key, None)
        source = doc.metadata.get("source", "N/A")

        print("=" * 100)
        print(f"Rank: {idx}")
        print(f"Source: {source}")
        if score is not None:
            print(f"Reranker score: {score:.6f}")
        print("-" * 100)
        print(doc.page_content[:max_chars].strip())
        print()


# ============================================================
# Two-stage retriever
# ============================================================

class TwoStageRetriever:
    """
    Two-stage retrieval pipeline:
    1. Dense retrieval from Chroma using sentence-transformers embeddings
    2. Cross-encoder reranking using sentence-transformers CrossEncoder

    This design keeps LangChain for orchestration and storage integration,
    while using the most stable Hugging Face-native path for reranking.
    """

    def __init__(
        self,
        vectorstore: Chroma,
        reranker_model_name: str,
        initial_k: int = 20,
        final_k: int = 5,
        device: Optional[str] = None,
        reranker_batch_size: int = 8,
    ) -> None:
        self.vectorstore = vectorstore
        self.initial_k = initial_k
        self.final_k = final_k
        self.device = get_device(device)
        self.reranker_batch_size = reranker_batch_size

        self.cross_encoder = CrossEncoder(
            reranker_model_name,
            device=self.device,
        )

    def _dense_retrieve(self, query: str) -> List[Document]:
        return self.vectorstore.similarity_search(query, k=self.initial_k)

    def _rerank(self, query: str, docs: Sequence[Document]) -> List[Document]:
        if not docs:
            return []

        pairs = [(query, doc.page_content) for doc in docs]

        scores = self.cross_encoder.predict(
            pairs,
            batch_size=self.reranker_batch_size,
            show_progress_bar=False,
        )

        reranked_docs: List[Document] = []
        for doc, score in zip(docs, scores):
            metadata = dict(doc.metadata) if doc.metadata else {}
            metadata["reranker_score"] = float(score)

            reranked_docs.append(
                Document(
                    page_content=doc.page_content,
                    metadata=metadata,
                )
            )

        reranked_docs.sort(
            key=lambda d: d.metadata["reranker_score"],
            reverse=True,
        )

        return reranked_docs[: self.final_k]

    def invoke(self, query: str) -> List[Document]:
        initial_docs = self._dense_retrieve(query)
        final_docs = self._rerank(query, initial_docs)
        return final_docs

    def batch(self, queries: Sequence[str]) -> List[List[Document]]:
        return [self.invoke(query) for query in queries]


# ============================================================
# Build vector store
# ============================================================

def build_embeddings(config: RetrievalConfig) -> HuggingFaceEmbeddings:
    device = get_device(config.device)

    return HuggingFaceEmbeddings(
        model_name=config.embedding_model_name,
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": config.normalize_embeddings},
    )


def build_vectorstore(config: RetrievalConfig) -> Chroma:
    embeddings = build_embeddings(config)

    vectorstore = Chroma(
        collection_name=config.collection_name,
        embedding_function=embeddings,
        persist_directory=config.persist_directory,
    )

    return vectorstore


# ============================================================
# Optional ingestion helper
# ============================================================

def add_documents_to_vectorstore(
    vectorstore: Chroma,
    texts: Sequence[str],
    metadatas: Optional[Sequence[Dict[str, Any]]] = None,
) -> None:
    docs = []
    if metadatas is None:
        metadatas = [{} for _ in texts]

    for text, metadata in zip(texts, metadatas):
        docs.append(Document(page_content=text, metadata=metadata))

    vectorstore.add_documents(docs)


# ============================================================
# Optional LCEL-style wrapper
# ============================================================

def build_lcel_rerank_runnable(two_stage_retriever: TwoStageRetriever):
    """
    Optional helper if you want to plug the retriever into a Runnable-based flow later.
    This keeps the implementation interoperable with the LangChain 1.x style.
    """
    from langchain_core.runnables import RunnableLambda

    return RunnableLambda(two_stage_retriever.invoke)


# ============================================================
# Example usage
# ============================================================

if __name__ == "__main__":
    config = RetrievalConfig(
        embedding_model_name="sentence-transformers/all-MiniLM-L6-v2",
        reranker_model_name="cross-encoder/ms-marco-MiniLM-L-6-v2",
        persist_directory="./chroma_langchain_db",
        collection_name="basic_ir_collection",
        initial_k=20,
        final_k=5,
        device=None,               # auto-detects cuda if available
        normalize_embeddings=True,
        reranker_batch_size=8,
    )

    print(f"Using device: {get_device(config.device)}")

    vectorstore = build_vectorstore(config)

    # --------------------------------------------------------
    # OPTIONAL: only run this once if your collection is empty
    # --------------------------------------------------------
    sample_texts = [
        "Autoencoders are neural networks trained to reconstruct input data.",
        "U-Nets are encoder-decoder architectures widely used in image segmentation.",
        "Cross-encoders score query-document pairs jointly and are often used for reranking.",
        "Bi-encoders independently embed queries and documents into a shared vector space.",
        "Two-stage retrieval combines fast dense retrieval with slower but more accurate reranking.",
    ]

    sample_metadatas = [
        {"source": "doc_1"},
        {"source": "doc_2"},
        {"source": "doc_3"},
        {"source": "doc_4"},
        {"source": "doc_5"},
    ]

    add_documents_to_vectorstore(vectorstore, sample_texts, sample_metadatas)

    retriever = TwoStageRetriever(
        vectorstore=vectorstore,
        reranker_model_name=config.reranker_model_name,
        initial_k=config.initial_k,
        final_k=config.final_k,
        device=config.device,
        reranker_batch_size=config.reranker_batch_size,
    )

    query = "What is the difference between bi-encoders and cross-encoders in retrieval pipelines?"
    results = retriever.invoke(query)

    print_ranked_results(results)

    # Optional LCEL-compatible runnable
    # runnable_retriever = build_lcel_rerank_runnable(retriever)
    # runnable_results = runnable_retriever.invoke(query)

Using device: cpu


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

Rank: 1
Source: doc_3
Reranker score: 1.145402
----------------------------------------------------------------------------------------------------
Cross-encoders score query-document pairs jointly and are often used for reranking.

Rank: 2
Source: doc_4
Reranker score: 1.021520
----------------------------------------------------------------------------------------------------
Bi-encoders independently embed queries and documents into a shared vector space.

Rank: 3
Source: doc_1
Reranker score: -4.071648
----------------------------------------------------------------------------------------------------
Autoencoders are neural networks trained to reconstruct input data.

Rank: 4
Source: doc_5
Reranker score: -4.425823
----------------------------------------------------------------------------------------------------
Two-stage retrieval combines fast dense retrieval with slower but more accurate reranking.

Rank: 5
Source: doc_2
Reranker score: -4.578892
-----------------------------

# 2-stage IR with eval metrics

One problem I have found in my research is evaluating LangChain IR. While many sources advised making granular evaluations of each component of the IR/RAG system and there was a lot of material on classic IR metrics, there wasn't a clear path on how to bring them into production with LangChain.

What I mean by that is that while calculating MRR/NDCG/etc-@k with a static gold standard is trivial, how exactly do we account for the specific challenges of a textual IR pipeline? How do evaluate the impact of our chosen text splitting/chunking method and parameters if that splitting changes the very annotations it's supposed to be evaluated on (since different splits may change the chunk a relevant information is placed in)?

After some more research and seeing some of the possible approaches to achieve the objectives I had set - getting classic IR metrics with a "chunking-agnostic" gold standard - I have reached a solution that satisfies my initial needs. Though how usable such a solution really is in a real-world production environment still remains to be seen.

All the code was vibe-coded initially, building on the original (also vibe-coded) 2-stage pipeline with no `langchain-classic` dependencies. But since I was unable to verify whether the metrics calculated were accurate or not, I ended up implementing the evaluation pipeline myself, doing many sanity-checks along the way. The original 2-stage IR pipeline and the `doccano` annotation converter are the only fully LLM-generated code remaining.

The base process to get these metrics is to get [`doccano`](https://github.com/doccano/doccano)-style span annotations that enable gold standard ground-truth query-chunk(s) pairs to be found regardless of how chunking splits the text. The `doccano` annotations are converted into a [BEIR](https://github.com/beir-cellar/beir)-style annotations, as the original plan was to use that package for the metric calculation, but I ended up deciding it was too much of a hassle for what it was worth. Once converted, the query-chunk pairs are used to calculate metrics (for now they are all zero because of the low amount of annotated examples I have, but I have tested with more samples and the implementations seem to result the expected results).

In [2]:
import langchain, langchain_core
langchain.__version__, langchain_core.__version__

('1.2.13', '1.2.19')

In [3]:
# @title Load dummy data source and gold standard annotations

!curl -o glossary.md https://raw.githubusercontent.com/GabrielvanderSchmidt/SENAC-DLGlossary/refs/heads/main/glossary.md
!gdown 13Ua7uHBjLJQLuz-cZIFD7V6u38w24LBs
!mkdir glossary
!mv glossary.md glossary/glossary.md

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 13632  100 13632    0     0  31949      0 --:--:-- --:--:-- --:--:-- 31925
Downloading...
From: https://drive.google.com/uc?id=13Ua7uHBjLJQLuz-cZIFD7V6u38w24LBs
To: /content/doccano_annotations.jsonl
100% 14.2k/14.2k [00:00<00:00, 26.2MB/s]


In [4]:
# Made with AdaptaOne
#
# @title Convert `doccano` annotations


import json
from pathlib import Path
from typing import Dict, List, Tuple

INPUT_JSONL = "doccano_annotations.jsonl"
OUTPUT_DIR = "beir_dataset"


def normalize_query(text: str) -> str:
    return " ".join(text.strip().split())


def load_doccano_jsonl(path: str) -> List[dict]:
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid value in line {line_number} in JSONL: {e}") from e
    return records


def convert_doccano_to_beir(records: List[dict], output_dir: str) -> None:
    output_path = Path(output_dir)
    output_path.mkdir(parents=True, exist_ok=True)

    corpus_path = output_path / "corpus.jsonl"
    queries_path = output_path / "queries.jsonl"
    qrels_path = output_path / "qrels.tsv"
    spans_path = output_path / "spans.jsonl"

    query_text_to_id: Dict[str, str] = {}
    queries_out: List[dict] = []
    corpus_out: List[dict] = []
    spans_out: List[dict] = []

    qrels_set: set[Tuple[str, str, int]] = set()

    query_counter = 1

    for record in records:
        doc_id = str(record["id"])
        doc_text = record.get("text", "")
        labels = record.get("label", [])

        corpus_out.append({
            "_id": doc_id,
            "title": "",
            "text": doc_text
        })

        for item in labels:
            if not isinstance(item, list) or len(item) != 3:
                continue

            start, end, query_text = item

            if not isinstance(start, int) or not isinstance(end, int):
                continue
            if not isinstance(query_text, str):
                continue
            if start < 0 or end > len(doc_text) or start >= end:
                continue

            normalized_query = normalize_query(query_text)
            if not normalized_query:
                continue

            if normalized_query not in query_text_to_id:
                query_id = f"q_{query_counter:06d}"
                query_text_to_id[normalized_query] = query_id
                queries_out.append({
                    "_id": query_id,
                    "text": normalized_query
                })
                query_counter += 1
            else:
                query_id = query_text_to_id[normalized_query]

            span_text = doc_text[start:end]

            qrels_set.add((query_id, doc_id, 1))

            spans_out.append({
                "query_id": query_id,
                "doc_id": doc_id,
                "start_index": start, # "start": start,
                "end_index": end, # "end": end,
                "page_content": span_text # "span_text": span_text
            })

    with open(corpus_path, "w", encoding="utf-8") as f:
        for row in corpus_out:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    with open(queries_path, "w", encoding="utf-8") as f:
        for row in queries_out:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    with open(qrels_path, "w", encoding="utf-8") as f:
        f.write("query-id\tcorpus-id\tscore\n")
        for query_id, doc_id, score in sorted(qrels_set):
            f.write(f"{query_id}\t{doc_id}\t{score}\n")

    with open(spans_path, "w", encoding="utf-8") as f:
        for row in spans_out:
            f.write(json.dumps(row, ensure_ascii=False) + "\n")

    print("Conversion complete.")
    print(f"Corpus:  {corpus_path}")
    print(f"Queries: {queries_path}")
    print(f"Qrels:   {qrels_path}")
    print(f"Spans:   {spans_path}")
    print(f"Total documents: {len(corpus_out)}")
    print(f"Total unique queries: {len(queries_out)}")
    print(f"Total relevant query-doc pairs: {len(qrels_set)}")
    print(f"Total spans annotaded: {len(spans_out)}")


if __name__ == "__main__":
    records = load_doccano_jsonl(INPUT_JSONL)
    convert_doccano_to_beir(records, OUTPUT_DIR)


Conversion complete.
Corpus:  beir_dataset/corpus.jsonl
Queries: beir_dataset/queries.jsonl
Qrels:   beir_dataset/qrels.tsv
Spans:   beir_dataset/spans.jsonl
Total documents: 1
Total unique queries: 4
Total relevant query-doc pairs: 4
Total spans annotaded: 7


In [6]:
# Made with the help of AdaptaOne
#
# @title Two-stage Information Retrieval Pipeline with Metrics

from __future__ import annotations

from dataclasses import dataclass, field
from typing import Sequence, Optional, Any
import hashlib
import os
import warnings

import numpy as np
import torch
from sentence_transformers import CrossEncoder
from transformers import AutoTokenizer

from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

# ============================================================
# Configuration
# ============================================================

@dataclass
class RetrievalConfig:
    # Embedding model for stage 1
    embedding_model_name: str = "sentence-transformers/all-MiniLM-L6-v2"

    # Cross-encoder reranker model for stage 2
    # Good lightweight default for limited consumer GPUs
    reranker_model_name: str = "cross-encoder/ms-marco-MiniLM-L-6-v2"

    # Chroma persistence
    persist_directory: str = "./chroma_langchain_db"
    collection_name: str = "basic_ir_collection"

    # Retrieval sizes
    initial_k: int = 20   # retrieved from vector store
    final_k: int = 5      # kept after reranking

    # Device and performance
    device: Optional[str] = None
    normalize_embeddings: bool = True

    # Cross-encoder inference batch size
    # Keep conservative for consumer GPU with limited VRAM
    reranker_batch_size: int = 8

    # Chunking/Splitting parameters
    chunk_size: int = 1000
    chunk_overlap: int = 200

    eval: bool = False
    at_k_ks: list[int] = field(default_factory=lambda: [5, 10, 20])

# ============================================================
# Utility functions
# ============================================================

def get_device(explicit_device: Optional[str] = None) -> str:
    if explicit_device:
        return explicit_device
    return "cuda" if torch.cuda.is_available() else "cpu"


def print_ranked_results(
    docs: Sequence[Document],
    score_key: str = "reranker_score",
    max_chars: int = 400,
    ) -> None:
    if not docs:
        print("No documents found.")
        return

    for idx, doc in enumerate(docs, start=1):
        score = doc.metadata.get(score_key, None)
        source = doc.metadata.get("source", "N/A")

        print("=" * 100)
        print(f"Rank: {idx}")
        print(f"Source: {source}")
        if score is not None:
            print(f"Reranker score: {score:.6f}")
        print("-" * 100)
        print(doc.page_content[:max_chars].strip())
        print()

# ============================================================
# Two-stage retriever
# ============================================================

class TwoStageRetriever:
    """
    Two-stage retrieval pipeline:
    1. Dense retrieval from Chroma using sentence-transformers embeddings
    2. Cross-encoder reranking using sentence-transformers CrossEncoder

    This design keeps LangChain for orchestration and storage integration,
    while using the most stable Hugging Face-native path for reranking.
    """

    def __init__(
        self,
        vectorstore: Chroma,
        reranker_model_name: str,
        initial_k: int = 20,
        final_k: int = 5,
        device: Optional[str] = None,
        reranker_batch_size: int = 8,
    ) -> None:
        self.vectorstore = vectorstore
        self.initial_k = initial_k
        self.final_k = final_k
        self.device = get_device(device)
        self.reranker_batch_size = reranker_batch_size

        self.cross_encoder = CrossEncoder(
            reranker_model_name,
            device=self.device,
        )

    def _dense_retrieve(self, query: str) -> list[Document]:
        return self.vectorstore.similarity_search(query, k=self.initial_k)

    def _rerank(self, query: str, docs: Sequence[Document]) -> list[Document]:
        if not docs:
            return []

        pairs = [(query, doc.page_content) for doc in docs]

        scores = self.cross_encoder.predict(
            pairs,
            batch_size=self.reranker_batch_size,
            show_progress_bar=False,
        )

        reranked_docs: list[Document] = []
        for doc, score in zip(docs, scores):
            metadata = dict(doc.metadata) if doc.metadata else {}
            metadata["reranker_score"] = float(score)

            reranked_docs.append(
                Document(
                    page_content=doc.page_content,
                    metadata=metadata,
                )
            )

        reranked_docs.sort(
            key=lambda d: d.metadata["reranker_score"],
            reverse=True,
        )

        return reranked_docs[: self.final_k]

    def invoke(self, query: str) -> list[Document]:
        initial_docs = self._dense_retrieve(query)
        final_docs = self._rerank(query, initial_docs)
        return final_docs

    def batch(self, queries: Sequence[str]) -> list[list[Document]]:
        return [self.invoke(query) for query in queries]

# ============================================================
# Build vector store
# ============================================================

def build_embeddings(config: RetrievalConfig) -> HuggingFaceEmbeddings:
    device = get_device(config.device)

    return HuggingFaceEmbeddings(
        model_name=config.embedding_model_name,
        model_kwargs={"device": device},
        encode_kwargs={"normalize_embeddings": config.normalize_embeddings},
    )


def build_vectorstore(config: RetrievalConfig) -> Chroma:
    embeddings = build_embeddings(config)

    vectorstore = Chroma(
        collection_name=config.collection_name,
        embedding_function=embeddings,
        persist_directory=config.persist_directory,
    )

    return vectorstore

# ============================================================
# Ingestion helpers
# ============================================================

def add_strings_to_vectorstore(
    vectorstore: Chroma,
    texts: Sequence[str],
    metadatas: Optional[Sequence[dict[str, Any]]] = None,
    ) -> None:
    docs = []
    if metadatas is None:
        metadatas = [{} for _ in texts]

    for text, metadata in zip(texts, metadatas):
        docs.append(Document(page_content=text, metadata=metadata))

    vectorstore.add_documents(docs)

def add_chunks_to_vectorstore(
        chunks: list[Document],
        vectorstore : Chroma,
        ) -> None:
    text_chunk = [chunk.page_content for chunk in chunks]
    encoded_chunks = [chunk.encode("utf-8") for chunk in text_chunk]
    ids = [hashlib.md5(chunk).hexdigest() for chunk in encoded_chunks]

    vectorstore.add_documents(documents=chunks, ids=ids)

def load_plain_text_documents(
        dir_path: Optional[str] = ".",
        custom_metadata: Optional[dict[str, Any]] = None,
        ) -> list[Document]:
    documents = [os.path.join(dir_path, file) for file in os.listdir(dir_path) if os.path.isfile(os.path.join(dir_path, file))]
    documents = [file for file in documents if os.path.isfile(file)]
    if len(documents) == 0:
        raise ValueError(f"No files found in {dir_path}.")
    documents_data = []
    documents_meta = []
    i = 0
    for doc in documents:
        with open(doc, "r") as file:
            base_meta = {"document_id" : str(i), "source" : doc}
            documents_data.append(file.read())
            documents_meta.append(base_meta if custom_metadata is None else base_meta | custom_metadata)
            i = i+1
    documents = [Document(page_content=text, metadata=meta) for text, meta in zip(documents_data, documents_meta)]
    return documents

def get_text_splitter(
        chunk_size: Optional[int] = 1000,
        chunk_overlap: Optional[int] = 200,
        ) -> RecursiveCharacterTextSplitter:
    # Using non-`from_huggingface_tokenizer` version for accurate `start_index` values
    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, # Chunk size (characters) -- we can't tell if it's smaller than embedder context length
        chunk_overlap=chunk_overlap, # Chunk overlap (characters) so context isn't lost at the "edges" of chunks
        add_start_index=True, # Track indexes in original document
        )

def load_ground_truth_spans(dataset_path: Optional[str] = "beir_dataset") -> list[Document]:
    docs : list[Document] = []
    with open(os.path.join(dataset_path, "spans.jsonl"), "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                span = json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid value in line {line_number} in JSONL: {e}") from e
            doc = Document(
                page_content=span.pop("page_content"),
                metadata=span
                )
            docs.append(doc)
    return docs

def load_ground_truth_queries(dataset_path: Optional[str] = "beir_dataset") -> dict[str, str]:
    queries : dict[str, str] = {}
    with open(os.path.join(dataset_path, "queries.jsonl"), "r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                query = json.loads(line)
            except json.JSONDecodeError as e:
                raise ValueError(f"Invalid value in line {line_number} in JSONL: {e}") from e
            queries[query["_id"]] = query["text"]
    return queries

def annotate_relevant_docs(chunk_ids: list[str],
                           chunks: list[Document],
                           related_query_ids: list[str],
                           gold_standard: list[Document]) -> dict[str, dict[str, float]]:
    relevant_docs : dict[str, dict] = dict()
    raw_relevances = get_span_ious(chunks, gold_standard)

    for i, query_id in enumerate(related_query_ids):
        # Apply softmax to IoU values, generating a relevance score that adds up to 1 IF span was broken
        relevances = softmax(raw_relevances[i])
        annotations = {id : relevance for id, relevance in zip(chunk_ids, relevances) if relevance != 0}

        # Extend annotations instead of overwriting, as queries may have multiple relevant spans
        queries_annotated = relevant_docs.get(query_id, None)
        if queries_annotated is None:
            relevant_docs[query_id] = annotations
        else:
            relevant_docs[query_id].update(annotations)
    return relevant_docs

# ============================================================
# Optional LCEL-style wrapper
# ============================================================

def build_lcel_rerank_runnable(two_stage_retriever: TwoStageRetriever):
    """
    Optional helper if you want to plug the retriever into a Runnable-based flow later.
    This keeps the implementation interoperable with the LangChain 1.x style.
    """
    from langchain_core.runnables import RunnableLambda

    return RunnableLambda(two_stage_retriever.invoke)

# ============================================================
# Text Splitting Metrics
# ============================================================

def get_chunk_indexes(chunk: Document,
                      infer_end_if_missing: Optional[bool] = True) -> tuple[int, int]:
    # If chunk has no start_index, let it error out
    chunk_start = chunk.metadata["start_index"]
    if chunk_start < 0: # When splitter fails to generate start_index, it defaults to -1
        warnings.warn(f"Invalid value found for chunk start index: {chunk_start}", category=RuntimeWarning, stacklevel=2)
    if infer_end_if_missing:
        chunk_end = chunk.metadata.get("end_index", chunk_start+len(chunk.page_content)) # We can infer end_index from start_index...
    else:
        chunk_end = chunk.metadata["end_index"] # ...or let it error out if we don't want to
    return chunk_start, chunk_end

# Measure chunk length in tokens
def token_chunk_len(chunk: str,
                    tokenizer: AutoTokenizer,
                    **tokenizer_kwargs) -> int:
    return len(tokenizer(chunk.page_content, **tokenizer_kwargs)['input_ids'])

def get_chunk_lengths(chunks: list[Document],
                      tokenizer: AutoTokenizer,
                      **tokenizer_kwargs) -> list[tuple[int, int]]:
    """ Get chunk lengths in characters and tokens """
    return [(len(chunk.page_content), token_chunk_len(chunk, tokenizer, **tokenizer_kwargs)) for chunk in chunks]

# Measure chunk/gold-standard IoU
def chunk_iou(chunk: Document,
              gold_span: Document) -> float:
    chunk_start, chunk_end = get_chunk_indexes(chunk)
    if chunk_start < 0: # If invalid index
        return 0.0
    # If gold standard has no start/end_index something went wrong, let it error out
    span_start, span_end = get_chunk_indexes(gold_span, infer_end_if_missing=False)

    intersection = min(chunk_end, span_end) - max(chunk_start, span_start)
    if intersection < 0:
        return 0.0
    union = max(chunk_end, span_end) - min(chunk_start, span_start)
    return intersection / union

def get_span_ious(chunks: list[Document],
                  gold_standard: list[Document]) -> list[list[float]]:
    ious : list[list[float]] = []
    for span in gold_standard:
        # Get IoU of all chunks
        iou = [chunk_iou(chunk, span) for chunk in chunks]
        ious.append(iou)
    return ious

def basic_range_metrics(values: list[int],
                        ignore_zeros: bool = False) -> dict[str, float]:
    """ Basic range metrics (can be replaced with plots or more detailed metrics) """
    if ignore_zeros:
        values = [v for v in values if v != 0]
    if len(values) == 0:
        if ignore_zeros:
            raise ValueError("`values` has no non-zero values")
        else:
            raise ValueError("`values` is empty")
    return {"max" : max(values),
            "min" : min(values),
            "mean" : sum(values)/len(values)}

def print_chunk_metrics(gt_spans: list[Document],
                        chunks: list[Document],
                        tokenizer: AutoTokenizer) -> None:
    # Len metrics
    lens = get_chunk_lengths(chunks, tokenizer)
    char_lens = [l[0] for l in lens]
    token_lens = [l[1] for l in lens]
    char_metrics = basic_range_metrics(char_lens)
    print(f"Chunk lengths measured in characters:\nmin = {char_metrics['min']:.2f}\t\tmax = {char_metrics['max']:.2f}\t\tmean = {char_metrics['mean']:.2f}")
    token_metrics = basic_range_metrics(token_lens)
    print(f"Chunk lengths measured in tokens:\nmin = {token_metrics['min']:.2f}\t\tmax = {token_metrics['max']:.2f}\t\tmean = {token_metrics['mean']:.2f}")

    # IoU
    ious = get_span_ious(chunks, gt_spans)
    # Keep only highest IoU per-gold-standard-span
    ious = [max(iou) for iou in ious]
    iou_metrics = basic_range_metrics(ious)
    print(f"Max IoUs of ground-truth answer spans with vectorbase chunks:\nmin = {iou_metrics['min']:.2f}\t\tmax = {iou_metrics['max']:.2f}\t\tmean = {iou_metrics['mean']:.2f}")

# ============================================================
# IR Metrics
# ============================================================

# Calculate chunk relevance per query from ground-truth IoU
def softmax(values: list[float]) -> list[float]:
    values = np.array(values, dtype=float)
    values = np.where(values == 0, float("-inf"), values)
    values = np.exp(values)
    return [float(x) for x in (values / sum(values))]

def get_docs_from_ids(ids: list[str]) -> list[Document]:
    return vectorstore.get_by_ids(ids)

def get_hits(retrieved_docs_at_i : list[Document],
             query_ground_truth : dict[str, float],
             return_scores : Optional[bool] = False
             ) -> list[bool] | list[float]:
    retrieved_ids = [doc.id for doc in retrieved_docs_at_i]
    if return_scores:
        return [query_ground_truth.get(id, 0.0) for id in retrieved_ids] # Return relevance scores
    else:
        return [bool(query_ground_truth.get(id, 0.0)) for id in retrieved_ids] # Return hits

# Mean Reciprocal Rank at K (MRR@K)
def mrr_at_k(query_ids: list[str],
             retrieved_docs: list[list[Document]],
             relevance_ground_truth: dict[str, list[tuple[str, float]]],
             ks: Optional[list[int]] = None) -> dict[int, float]:
    if ks is None:
        ks = [5, 10, 20]
    mrrs : dict[int, float] = dict()
    for k in ks:
        rrs : list[float] = []
        for i, query_id in enumerate(query_ids):
            hits = get_hits(retrieved_docs[i][:k], relevance_ground_truth[query_id], return_scores=False)
            hit_index = np.where(hits)[0]
            rr = 0.0 if len(hit_index) == 0 else 1.0 / (hit_index[0] + 1)
            rrs.append(rr)
        mrrs[k] = float(np.mean(rrs))
    return mrrs

# Mean Precision at K
def precision_at_k(query_ids: list[str],
                   retrieved_docs: list[list[Document]],
                   relevance_ground_truth: dict[str, list[tuple[str, float]]],
                   ks: Optional[list[int]] = None) -> dict[int, float]:
    if ks is None:
        ks = [5, 10, 20]
    mean_precisions : dict[int, float] = dict()
    for k in ks:
        precisions : list[float] = []
        for i, query_id in enumerate(query_ids):
            hits = get_hits(retrieved_docs[i][:k], relevance_ground_truth[query_id], return_scores=False)
            precisions.append(sum(hits) / k)
        mean_precisions[k] = float(np.mean(precisions))
    return mean_precisions

# Mean Recall at K
def recall_at_k(query_ids: list[str],
                   retrieved_docs: list[list[Document]],
                   relevance_ground_truth: dict[str, list[tuple[str, float]]],
                   ks: Optional[list[int]] = None) -> dict[int, float]:
    if ks is None:
        ks = [5, 10, 20]
    mean_precisions : dict[int, float] = dict()
    for k in ks:
        precisions : list[float] = []
        for i, query_id in enumerate(query_ids):
            hits = get_hits(retrieved_docs[i][:k], relevance_ground_truth[query_id], return_scores=False)
            precisions.append(sum(hits) / len(relevance_ground_truth[query_id]))
        mean_precisions[k] = float(np.mean(precisions))
    return mean_precisions

# Mean Average Precision at K (mAP@K)
# Implemented according to the explanaiton in:
    # https://towardsdatascience.com/mean-average-precision-at-k-map-k-clearly-explained-538d8e032d2/
def average_precision_at_k(hits : list[bool],
                           k : int) -> float:
    # Calculate precisions from 1 to k to get average precision
    average_precision : float = 0.0
    for i in range(1, k + 1):
        average_precision += (sum(hits[:i]) / i) * hits[i-1]
    n_relevant = sum(hits)
    return average_precision / n_relevant if n_relevant > 0 else 0.0

def map_at_k(query_ids: list[str],
             retrieved_docs: list[list[Document]],
             relevance_ground_truth: dict[str, list[tuple[str, float]]],
             ks: Optional[list[int]] = None) -> dict[int, float]:
    if ks is None:
        ks = [5, 10, 20]
    maps : dict[int, float] = dict()
    for k in ks:
        aps : list[float] = []
        for i, query_id in enumerate(query_ids):
            hits = get_hits(retrieved_docs[i][:k], relevance_ground_truth[query_id], return_scores=False)
            aps.append(average_precision_at_k(hits, k))
        maps[k] = float(np.mean(aps))
    return maps

# Mean Normalized Discounted Cumulative Gain at K (Mean NDCG@K)
def dcg_at_k(scores : list[float],
             k : int) -> float:
    scores = np.array(scores)
    discounts = np.log2(np.arange(2, k + 2))
    return float(np.sum(scores / discounts))

def ndcg_at_k(query_ids: list[str],
              retrieved_docs: list[list[Document]],
              relevance_ground_truth: dict[str, list[tuple[str, float]]],
              ks: Optional[list[int]] = None) -> dict[int, float]:
    if ks is None:
        ks = [5, 10, 20]
    mean_ndcgs : dict[int, float] = dict()
    for k in ks:
        ndcgs : list[float] = []
        for i, query_id in enumerate(query_ids):
            scores = get_hits(retrieved_docs[i][:k], relevance_ground_truth[query_id], return_scores=True)
            # Get Discounted Cumulative Gain for retreived documents
            dcg = dcg_at_k(scores, k)
            # Get Ideal Discounted Cumulative Gain (max DCG score possible, if relevant items appear in descending relevance order)
            ideal_scores = [relevance_ground_truth[query_id][doc] for doc in relevance_ground_truth[query_id]] # Get all scores annotated
            ideal_scores.sort(reverse=True)
            if len(ideal_scores) < k:
                ideal_scores += [0.0] * (k - len(ideal_scores)) # Zero-fill if possible relevant chunks is less than k
            idcg = dcg_at_k(ideal_scores, k)
            ndcgs.append(dcg / idcg)
        mean_ndcgs[k] = float(np.mean(ndcgs))
    return mean_ndcgs

def print_ir_metrics(query_ids: list[str],
                     retrieved_docs: list[list[Document]],
                     relevance_ground_truth: dict[str, list[tuple[str, float]]],
                     ks: Optional[list[int]] = None) -> None:
    if ks is None:
        ks = [5, 10, 20]
    mrrs = mrr_at_k(query_ids, retrieved_docs, relevance_ground_truth, ks)
    mprecisions = precision_at_k(query_ids, retrieved_docs, relevance_ground_truth, ks)
    mrecalls = recall_at_k(query_ids, retrieved_docs, relevance_ground_truth, ks)
    maps = map_at_k(query_ids, retrieved_docs, relevance_ground_truth, ks)
    ndcg = ndcg_at_k(query_ids, retrieved_docs, relevance_ground_truth, ks)
    for k in ks:
        print(f"=== @k = {k} ===")
        print(f"MRR@{k} = {mrrs[k]:.4f}")
        print(f"Mean Precision@{k} = {mprecisions[k]:.4f}")
        print(f"Mean Recall@{k} = {mrecalls[k]:.4f}")
        print(f"MAP@{k} = {maps[k]:.4f}")
        print(f"Mean NDCG@{k} = {ndcg[k]:.4f}")
        print()

# ============================================================
# Example usage
# ============================================================

if __name__ == "__main__":
    config = RetrievalConfig(
        embedding_model_name="sentence-transformers/all-MiniLM-L6-v2",
        reranker_model_name="cross-encoder/ms-marco-MiniLM-L-6-v2",
        persist_directory="./chroma_langchain_db",
        collection_name="basic_ir_collection",
        initial_k=20,
        final_k=5,
        device=None,               # auto-detects cuda if available
        normalize_embeddings=True,
        reranker_batch_size=8,
        eval=True,
        at_k_ks=[1, 5],
    )

    print(f"Using device: {get_device(config.device)}")

    # Create and populate vectorstore
    vectorstore = build_vectorstore(config)
    documents = load_plain_text_documents("glossary")

    # Chunk text
    doc_splitter = get_text_splitter(config.chunk_size, config.chunk_overlap)
    chunks = doc_splitter.split_documents(documents)

    add_chunks_to_vectorstore(chunks, vectorstore)

    if config.eval:
        # Show splitting metrics on loaded docs
        ground_truth_spans = load_ground_truth_spans()
        tokenizer = AutoTokenizer.from_pretrained(config.embedding_model_name)
        print_chunk_metrics(ground_truth_spans, chunks, tokenizer)

        # Get chunk relevance values for IR metrics
        all_chunk_ids = vectorstore.get()["ids"]
        all_chunks = get_docs_from_ids(all_chunk_ids)

        ground_truth_spans = load_ground_truth_spans()
        related_query_ids = [query.metadata["query_id"] for query in ground_truth_spans]
        annotated_relevances = annotate_relevant_docs(all_chunk_ids, all_chunks, related_query_ids, ground_truth_spans)

    retriever = TwoStageRetriever(
        vectorstore=vectorstore,
        reranker_model_name=config.reranker_model_name,
        initial_k=config.initial_k,
        final_k=config.final_k,
        device=config.device,
        reranker_batch_size=config.reranker_batch_size,
    )

    if config.eval:
        # Pretend all chunks where retrieved for each query
        queries = load_ground_truth_queries()
        query_ids = queries.keys()
        query_texts = [queries[query_id] for query_id in query_ids]

        # Retrieve chunks for test queries
        results = retriever.batch(query_texts)
        print_ir_metrics(query_ids, results, annotated_relevances, ks=config.at_k_ks)

        print_ranked_results(results[0])
    else:
        # Display results of a single query just to show pipeline working
        query = ["Qual a diferença entre autoencoders e U-Nets?"]
        results = retriever.invoke(query)
        print_ranked_results(results)

    # Optional LCEL-compatible runnable
    # runnable_retriever = build_lcel_rerank_runnable(retriever)
    # runnable_results = runnable_retriever.invoke(query)

Using device: cpu


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Chunk lengths measured in characters:
min = 16.00		max = 992.00		mean = 658.43
Chunk lengths measured in tokens:
min = 10.00		max = 402.00		mean = 253.86
Max IoUs of ground-truth answer spans with vectorbase chunks:
min = 0.11		max = 0.57		mean = 0.27


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

=== @k = 1 ===
MRR@1 = 0.0000
Mean Precision@1 = 0.0000
Mean Recall@1 = 0.0000
MAP@1 = 0.0000
Mean NDCG@1 = 0.0000

=== @k = 5 ===
MRR@5 = 0.0000
Mean Precision@5 = 0.0000
Mean Recall@5 = 0.0000
MAP@5 = 0.0000
Mean NDCG@5 = 0.0000

Rank: 1
Source: glossary/glossary.md
Reranker score: 4.876695
----------------------------------------------------------------------------------------------------
# Estrutura da Rede


#### Perceptron (Neurônio Artificial)
É o tipo mais simples de rede neural **feedforward**, e hoje é usado como sinônimo para **neurônio artificial**, que é o bloco fundamental de **redes neurais artificiais**. Neurônios artificiais são agrupados lateralmente em **camadas densas**, e estas são empilhadas verticalmente para formar **redes neurais multicamadas**. Consiste de um

Rank: 2
Source: glossary/glossary.md
Reranker score: 0.170184
----------------------------------------------------------------------------------------------------
#### Rede Recursiva (Recursive Neural Ne